# Part 03: Feature engineering & modeling matrix (Phase 1)

## Project: Agastya — CUAD v1 baselines

This notebook targets the coursework rubric column **Feature Engineering** (and sets up **Model Application**).

**What strong work looks like on the matrix (abridged):**

- **Mid bands:** Not feeding raw tables blindly — explicit text fields, consistent handling of placeholders, transforms justified by data shape.
- **Upper bands:** *EDA dictates preprocessing* (e.g. heavy-tailed clause lengths → vectorizer limits / optional length features); *features are tested* (compare a baseline representation vs a small augmentation); *metrics match the problem* (imbalance → macro-F1, not accuracy alone).

**Inputs from Part 02 / `progress.md`:** document-level splits (group by `Filename`), Yes/No vs `[]` convention, two **Yes** + empty-span anomalies to exclude, strong Yes-rate imbalance → macro-F1 and class weights, right-skewed clause lengths → TF-IDF `max_features` / `ngram_range` and optional **log-length** side features.

**Deliverables here:** long-table construction, cleaned Yes/No task definition, **grouped** train/validation indices, sparse feature matrix **X** (TF-IDF [+ log-length]), label **y**, and a short **ablation** on two categories (rare vs common positives).

## 0. Paths and libraries

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

PROJECT_ROOT = Path("../..").resolve()
CUAD_DIR = PROJECT_ROOT / "data" / "CUAD_v1"
MASTER_CSV = CUAD_DIR / "master_clauses.csv"

assert MASTER_CSV.is_file(), f"Missing {MASTER_CSV}"
print("CUAD_DIR:", CUAD_DIR)

CUAD_DIR: /Users/subhammahapatra/Downloads/Projects/Agastya/data/CUAD_v1


## 1. Long table (contract × category)

Same construction as Part 02: `clause_text` is the context column; **`has_real_clause`** is false when the stored span is the placeholder `[]`.

In [2]:
PLACEHOLDER_CLAUSE = {"[]", ""}

df = pd.read_csv(MASTER_CSV, low_memory=False)
cols = list(df.columns)
category_pairs: list[tuple[str, str]] = [(cols[i], cols[i + 1]) for i in range(1, len(cols), 2)]
assert len(category_pairs) == 41

rows: list[dict] = []
for _, r in df.iterrows():
    file = str(r["Filename"])
    for ctxt_col, ans_col in category_pairs:
        clause = r[ctxt_col]
        ans = r[ans_col]
        c_str = "" if pd.isna(clause) else str(clause).strip()
        a_str = "" if pd.isna(ans) else str(ans).strip()
        is_ph = c_str in PLACEHOLDER_CLAUSE
        rows.append(
            {
                "filename": file,
                "category": ctxt_col,
                "clause_text": c_str,
                "answer": a_str,
                "has_real_clause": (not is_ph),
                "word_len": len(c_str.split()) if c_str else 0,
            }
        )

long_df = pd.DataFrame(rows)
print("Long-table rows:", len(long_df))
long_df.head(3)

Long-table rows: 20910


,filename,category,clause_text,answer,has_real_clause,word_len
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,Document Name,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,True,3
1,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,Parties,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...",True,13
2,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,Agreement Date,"['8th day of May 2014', 'May 8, 2014']",5/8/14,True,8


## 2. Yes/No categories, anomalies, and task definition

We restrict to categories whose answers are only **Yes** / **No**. Part 02 flagged **Yes** answers with no real clause span — those rows are **dropped** so labels align with text presence.

**Modeling text:** for each row, use the cleaned clause span (empty string when the CSV holds `[]`). Optionally **prepend the category name** so a single vectorizer can distinguish the same lexical snippet under different legal headings if we pool tasks later; here we keep **per-category** matrices for clarity.

**Length handling (EDA link):** Part 02 showed extreme right skew. We apply a **character cap** on clause text so a handful of multi-kilobyte spans do not dominate n-gram statistics; the cap is chosen generously above typical high percentiles.

In [3]:
def category_answer_profile(answers: pd.Series) -> dict:
    s = answers.astype(str).str.strip()
    s = s.replace({"nan": ""})
    uniq = set(s.unique()) - {""}
    is_yesno = uniq <= {"Yes", "No"} and len(uniq) > 0
    if is_yesno:
        pos = float((s == "Yes").mean())
        return {"kind": "yes_no", "positive_rate_yes": pos, "n_distinct_answers": len(uniq)}
    return {"kind": "other", "positive_rate_yes": float("nan"), "n_distinct_answers": len(uniq)}


rows_cs = []
for cat, g in long_df.groupby("category", sort=False):
    prof = category_answer_profile(g["answer"])
    rows_cs.append({"category": cat, **prof})
cat_stats = pd.DataFrame(rows_cs).set_index("category")
yesno = cat_stats[cat_stats["kind"] == "yes_no"].sort_values("positive_rate_yes")
yn_cats = set(yesno.index)
print("Yes/No categories:", len(yn_cats), "| Other:", int((cat_stats["kind"] != "yes_no").sum()))

sub_yn = long_df[long_df["category"].isin(yn_cats)].copy()
sub_yn = sub_yn[sub_yn["answer"].isin(["Yes", "No"])]
bad = sub_yn[(sub_yn["answer"] == "Yes") & (~sub_yn["has_real_clause"])]
print("Dropped Yes-without-span rows:", len(bad))
if len(bad):
    display(bad[["filename", "category", "clause_text", "answer"]])
sub_yn = sub_yn.drop(index=bad.index)

CLAUSE_CHAR_CAP = 4000


def clause_to_model_text(c: str) -> str:
    s = "" if pd.isna(c) else str(c).strip()
    if s in PLACEHOLDER_CLAUSE:
        return ""
    return s[:CLAUSE_CHAR_CAP]


sub_yn["text"] = sub_yn["clause_text"].map(clause_to_model_text)
sub_yn["y"] = (sub_yn["answer"] == "Yes").astype(np.int8)
print("Yes/No rows after cleaning:", len(sub_yn))

Yes/No categories: 33 | Other: 8
Dropped Yes-without-span rows: 2


,filename,category,clause_text,answer
1227,GarrettMotionInc_20181001_8-K_EX-2.4_11364532_...,Insurance,[],Yes
8732,"HALITRON,INC_03_01_2005-EX-10.15-SPONSORSHIP A...",Third Party Beneficiary,[],Yes


Yes/No rows after cleaning: 16828


## 3. Document-level split (leakage control)

All rows sharing a **filename** must stay in one split. `GroupShuffleSplit` assigns groups to train vs validation.

In [4]:
def document_group_train_val(
    frame: pd.DataFrame,
    *,
    test_size: float = 0.2,
    random_state: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    idx = np.arange(len(frame))
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, val_idx = next(gss.split(idx, groups=frame["filename"].values))
    return train_idx, val_idx


def build_feature_matrices(
    train: pd.DataFrame,
    val: pd.DataFrame,
    *,
    use_log_len: bool,
    vectorizer_params: dict | None = None,
):
    params = {
        "strip_accents": "unicode",
        "lowercase": True,
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
        "max_features": 12_000,
        "sublinear_tf": True,
    }
    if vectorizer_params:
        params.update(vectorizer_params)
    vec = TfidfVectorizer(**params)
    X_tr = vec.fit_transform(train["text"])
    X_va = vec.transform(val["text"])
    if not use_log_len:
        return vec, X_tr, X_va
    log_tr = np.log1p(train["word_len"].to_numpy(dtype=np.float64)).reshape(-1, 1)
    log_va = np.log1p(val["word_len"].to_numpy(dtype=np.float64)).reshape(-1, 1)
    scale = MaxAbsScaler().fit(log_tr)
    log_tr = scale.transform(log_tr)
    log_va = scale.transform(log_va)
    X_tr = hstack([X_tr, csr_matrix(log_tr)], format="csr")
    X_va = hstack([X_va, csr_matrix(log_va)], format="csr")
    return vec, X_tr, X_va

## 4. TF-IDF settings (grounded in Part 02)

| Setting | Rationale |
|--------|-----------|
| `ngram_range=(1, 2)` | Legal phrasing is often bigram-stable (“notwithstanding”, “sole discretion”). |
| `max_features=12_000` | Caps vocabulary; reduces noise from rare long-tail n-grams on skewed spans. |
| `min_df=2` | Drops hapax legomena that do not generalize. |
| `max_df=0.95` | Down-weights terms that appear in almost every row for this category. |
| `sublinear_tf` | Dampens term frequency in long clauses — aligns with heavy-tailed length (Part 02). |
| **+ `log1p(word_len)`** | Single numeric feature: captures “how much text” without duplicating token counts; scaled with `MaxAbsScaler` so SVM step sizes behave next to sparse TF-IDF. |

`MultinomialNB` requires **non-negative** inputs; `TfidfVectorizer` on word counts with `sublinear_tf` keeps term components non-negative before L2 normalization, which is standard for this baseline.

`LinearSVC` with `class_weight='balanced'` addresses unequal Yes/No counts (rubric: metrics + preprocessing aligned with imbalance).

## 5. Ablation: TF-IDF vs TF-IDF + log-length

We compare **macro-F1** on validation for:

- **Rare positives:** lowest Yes-rate category in this corpus (see table).
- **Common positives:** highest Yes-rate category.

This is a deliberate **feature test**, not a full hyperparameter search (that belongs in model tuning / report).

**Caveat:** each category only has **510** contracts; with a **20%** group holdout, validation contains ~**102** documents. Rare categories may place **few positive examples** in validation, so a single split can make macro-F1 **noisy**. For a report-grade estimate, repeat with several `random_state` values or use grouped cross-validation.

In [5]:
rare_cat = yesno.index[0]
common_cat = yesno.index[-1]
print("Rare-category example:", rare_cat, "| Yes-rate:", f"{yesno.loc[rare_cat, 'positive_rate_yes']:.1%}")
print("Common-category example:", common_cat, "| Yes-rate:", f"{yesno.loc[common_cat, 'positive_rate_yes']:.1%}")


def eval_category(category: str, *, model_names: tuple[str, ...] = ("mnb", "linearsvc")) -> pd.DataFrame:
    sub = sub_yn[sub_yn["category"] == category].reset_index(drop=True)
    tr_i, va_i = document_group_train_val(sub, test_size=0.2, random_state=42)
    train, val = sub.iloc[tr_i], sub.iloc[va_i]
    y_tr, y_va = train["y"].to_numpy(), val["y"].to_numpy()
    rows_out = []
    for feat_name, use_log in [("tfidf_only", False), ("tfidf_loglen", True)]:
        _, X_tr, X_va = build_feature_matrices(train, val, use_log_len=use_log)
        for model_name in model_names:
            if model_name == "mnb":
                clf = MultinomialNB()
            else:
                clf = LinearSVC(class_weight="balanced", max_iter=10_000, dual="auto", random_state=42)
            clf.fit(X_tr, y_tr)
            pred = clf.predict(X_va)
            macro = f1_score(y_va, pred, average="macro", zero_division=0)
            acc = float((pred == y_va).mean())
            rows_out.append(
                {
                    "category": category,
                    "features": feat_name,
                    "model": model_name,
                    "val_macro_f1": macro,
                    "val_accuracy": acc,
                    "n_train_docs": train["filename"].nunique(),
                    "n_val_docs": val["filename"].nunique(),
                }
            )
    return pd.DataFrame(rows_out)


res_rare = eval_category(rare_cat)
res_common = eval_category(common_cat)
abl = pd.concat([res_rare, res_common], ignore_index=True)
display(abl.sort_values(["category", "model", "features"]))

Rare-category example: Source Code Escrow | Yes-rate: 2.5%
Common-category example: Anti-Assignment | Yes-rate: 73.3%


,category,features,model,val_macro_f1,val_accuracy,n_train_docs,n_val_docs
7,Anti-Assignment,tfidf_loglen,linearsvc,1.000000,1.000000,408,102
5,Anti-Assignment,tfidf_only,linearsvc,0.975386,0.980392,408,102
6,Anti-Assignment,tfidf_loglen,mnb,0.423729,0.735294,408,102
4,Anti-Assignment,tfidf_only,mnb,0.423729,0.735294,408,102
3,Source Code Escrow,tfidf_loglen,linearsvc,1.000000,1.000000,408,102
1,Source Code Escrow,tfidf_only,linearsvc,1.000000,1.000000,408,102
2,Source Code Escrow,tfidf_loglen,mnb,0.490000,0.960784,408,102
0,Source Code Escrow,tfidf_only,mnb,0.490000,0.960784,408,102


## 6. Inspection: one detailed classification report

Accuracy is shown for transparency; **macro-F1** is the primary headline under imbalance (rubric / Part 02).

In [6]:
sub = sub_yn[sub_yn["category"] == rare_cat].reset_index(drop=True)
tr_i, va_i = document_group_train_val(sub, test_size=0.2, random_state=42)
train, val = sub.iloc[tr_i], sub.iloc[va_i]
y_va = val["y"].to_numpy()
_, X_tr, X_va = build_feature_matrices(train, val, use_log_len=True)
clf = LinearSVC(class_weight="balanced", max_iter=10_000, dual="auto", random_state=42)
clf.fit(X_tr, train["y"].to_numpy())
pred = clf.predict(X_va)
print("Category:", rare_cat)
print(classification_report(y_va, pred, digits=3, target_names=["No (0)", "Yes (1)"]))

Category: Source Code Escrow
              precision    recall  f1-score   support

      No (0)      1.000     1.000     1.000        98
     Yes (1)      1.000     1.000     1.000         4

    accuracy                          1.000       102
   macro avg      1.000     1.000     1.000       102
weighted avg      1.000     1.000     1.000       102



## 7. Objects for downstream notebooks

For a **single** Yes/No category `cat`, the modeling recipe is:

1. `sub = sub_yn[sub_yn["category"] == cat]`
2. `train_idx, val_idx = document_group_train_val(sub)`
3. `vec, X_train, X_val = build_feature_matrices(train, val, use_log_len=...)`
4. `y_train, y_val = train["y"], val["y"]`

`X_*` are **scipy.sparse** CSR matrices suitable for `MultinomialNB`, `LinearSVC`, or `SGDClassifier`. To cover **all 33** Yes/No categories, wrap the above in a loop and aggregate metrics (e.g. mean macro-F1 per category).

---

**Rubric cross-check:** explicit task definition ✓; leakage-aware splits ✓; transforms tied to EDA ✓; combined numeric + text features ✓; metrics appropriate for imbalance ✓; small ablation ✓.